# Reasoning Data Curator

This notebook implements a data curation pipeline for reasoning datasets. <br><br>It draws inspiration from this sample in our [NeMo Curator Github](https://github.com/NVIDIA-NeMo/Curator/tree/main/tutorials/llama-nemotron-data-curation).  Please see that notebook for extra details.<br>  Our Curator notebook it filters, processes, and organizes data for curriculum learning by:

1. **Filtering** samples based on various criteria (language, token count, formatting)
2. **Sorting** by completion token count 
3. **Splitting** into reasoning ON/OFF datasets
4. **Interleaving** the datasets for balanced training

The supporting code and filter classes are imported from `reasoning_run_curator.py`.

Not sure what curriculum learning is?  Its a method of starting with simple examples and working up to complex ones.  Thats how the Nemotron data is organized.


## Setup and Imports

Import all necessary libraries and custom functions from the supporting script.


In [ ]:
# Setup Environment
import os
from dotenv import load_dotenv
# Load the .env file
load_dotenv('/workspace/docker/.env', override=True)

print(f"Current working dir           : { os.getcwd() }")
print(f"lHF TOKEN for lab : {os.getenv('HF_TOKEN')}")
!nvidia-smi


In [ ]:
!huggingface-cli login --token {os.getenv('HF_TOKEN')}

In [ ]:
import argparse

import time
from itertools import zip_longest

import cudf
import dask.dataframe as dd
import dask_cudf
import fasttext
import pandas as pd
from dask.delayed import delayed
from dask_cuda import LocalCUDACluster
from transformers import AutoTokenizer

from nemo_curator import ScoreFilter, Sequential
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import DocumentFilter
from nemo_curator.utils.decorators import batched
from nemo_curator.utils.distributed_utils import (
    NoWorkerError,
    get_client,
    load_object_on_worker,
)
from nemo_curator.utils.file_utils import get_all_files_paths_under
from nemo_curator.utils.script_utils import ArgumentHelper

# Import all custom filters and functions ./scripts/reasoning_nemo_curator.py
from reasoning_nemo_curator import (
    NanoFilter,
    EmptyThinkTagsFilter,
    MalformedFilter,
    MissingThinkCloseTagFilter,
    ContainsThinkOpenTagFilter,
    MissingThinkOpenTagFilter,
    NonEnglishFilter,
    TokenCountFilter,
    CompletionTokenCountFilter,
    format_partition,
    interleave_partitions,
    interleave_rows,
    attach_args
)


## Configuration Parameters

Set up the configuration parameters for the data curation pipeline. These parameters control:
- Input/output directories
- Tokenizer and language model settings
- Token count limits
- Processing options


In [ ]:
# Configuration parameters (modify these as needed)
class Config:
    def __init__(self):
        # Input/Output paths
        self.input_dir = "/workspace/data/Llama-Nemotron-Post-Training-Dataset/SFT/chat"
        self.output_dir = "/workspace/data/curated-data"
        
        # File filtering
        self.filename_filter = None  # List of substrings to filter filenames, or None for all files
        self.remove_columns = None   # List of columns to remove, or None to keep all
        
        # Model and tokenizer settings
        self.tokenizer = "meta-llama/Llama-3.1-8B-Instruct"
        self.lang_id_model_path = "/workspace/models/lid.176.ftz"
        
        # Token count limits
        self.max_token_count = 16384
        self.max_completion_token_count = 8192
        
        # Processing options
        self.json_blocksize = "100mb"  # Can be string like "100mb" or None
        self.json_files_per_partition = None  # Number of files per partition, or None
        
        # Interleaving options
        self.global_interleave = False  # True for row-by-row, False for partition-by-partition
        
        # Statistics and monitoring
        self.generate_statistics = True  # Set to False for large datasets to improve performance
        
        # Distributed computing settings
        self.device = "gpu"  # "cpu" or "gpu"
        
        # Dask client settings (modify as needed)
        self.scheduler_address = None
        self.scheduler_file = None
        self.n_workers = 2
        self.threads_per_worker = None
        self.rmm_pool_size = None

# Create config instance
config = Config()

# Display current configuration
print("Current Configuration:")
for attr, value in vars(config).items():
    print(f"  {attr}: {value}")


In [ ]:
# Clear Curated Data Directory if previously run
!rm -rf {config.output_dir}/*.jsonl {config.output_dir}/*.info {config.output_dir}/*.npy
!tree {config.input_dir}
!tree {config.output_dir}


## Distributed Computing Setup

Initialize the Dask client for distributed computing. This enables parallel processing across multiple workers.


In [ ]:
# Convert config to argparse-like namespace for compatibility
class Args:
    def __init__(self, config):
        for attr, value in vars(config).items():
            setattr(self, attr, value)

args = Args(config)

# Initialize Dask client
client_args = {
    'cluster_type' : args.device,
    'scheduler_address': args.scheduler_address,
    'scheduler_file': args.scheduler_file,
    'n_workers': args.n_workers,
    'threads_per_worker': args.threads_per_worker,
    'rmm_pool_size': args.rmm_pool_size,
}

# Filter out None values
client_args = {k: v for k, v in client_args.items() if v is not None}
client_args



In [ ]:
# https://github.com/NVIDIA-NeMo/Curator/blob/main/nemo_curator/utils/distributed_utils.py
client = get_client(**client_args)
print(f"Dask client initialized: {client}")

# Start timing
start_time = time.time()
print(f"Starting data curation pipeline at {time.ctime()}")

In [ ]:
client

In [ ]:
client.cluster

## Data Loading

Load JSONL files from the input directory. The pipeline will:
1. Find all JSONL files in the input directory
2. Apply filename filtering (if specified)
3. Read files into a Dask DataFrame with appropriate partitioning


In [ ]:
# check number of reasoning on vs reasoning off
!grep '"reasoning": "off"' {config.input_dir}/chat_10k.jsonl | wc -l
!grep '"reasoning": "on"' {config.input_dir}/chat_10k.jsonl | wc -l

In [ ]:
# Handle input path
input_files = list(get_all_files_paths_under(args.input_dir, keep_extensions="jsonl"))
print(f"Found {len(input_files)} total JSONL files")

if args.filename_filter:
    # Filter out files that don't contain any of the provided substrings
    input_files = [filename for filename in input_files if any(s in filename for s in args.filename_filter)]
    print(f"After filtering, using {len(input_files)} files:")
    for f in input_files:
        print(f"  - {f}")

# If neither is set, set the default blocksize to 100MB
if args.json_blocksize is None and args.json_files_per_partition is None:
    args.json_blocksize = "100mb"

print(f"Reading JSONL files with blocksize {args.json_blocksize} and files_per_partition {args.json_files_per_partition}")
dataset = DocumentDataset.read_json(
    input_files, blocksize=args.json_blocksize, files_per_partition=args.json_files_per_partition
)

print(f"Dataset loaded with {dataset.df.npartitions} partitions")
print(f"Dataset columns: {list(dataset.df.columns)}")


## Initial Statistics

Generate initial statistics about the dataset before filtering. This helps track how many samples are removed during the filtering process.


In [ ]:
if args.generate_statistics:
    print("Generating initial statistics...")
    initial_count = dataset.df.shape[0].compute()
    print(f"Initial number of samples: {initial_count:,}")
else:
    print("Skipping initial statistics generation for performance")
    initial_count = None


## Data Filtering and Token Counting

Apply a sequential pipeline of filters to clean and process the data. The filters include:

1. **NanoFilter**: Filters based on "nano" keyword in training usage
2. **EmptyThinkTagsFilter**: Removes samples with empty thinking tags
3. **MalformedFilter**: Filters malformed samples with boxed content issues
4. **MissingThinkCloseTagFilter**: Ensures proper closing of think tags
5. **ContainsThinkOpenTagFilter**: Handles reasoning off samples with think tags
6. **MissingThinkOpenTagFilter**: Ensures reasoning on samples have think tags
7. **NonEnglishFilter**: Filters non-English content using FastText
8. **TokenCountFilter**: Filters based on total token count
9. **CompletionTokenCountFilter**: Filters based on completion token count


In [ ]:
# Filter out samples based on token count
print("Applying filters and counting tokens...")

filter_steps = Sequential([
    ScoreFilter(
        NanoFilter(),
        text_field="used_in_training",
        score_type=bool,
    ),
    ScoreFilter(
        EmptyThinkTagsFilter(),
        text_field="output",
        score_type=bool,
    ),
    ScoreFilter(
        MalformedFilter(),
        text_field=["input", "output"],
        score_type=bool,
    ),
    ScoreFilter(
        MissingThinkCloseTagFilter(),
        text_field="output",
        score_type=bool,
    ),
    ScoreFilter(
        ContainsThinkOpenTagFilter(),
        text_field=["reasoning", "output"],
        score_type=bool,
    ),
    ScoreFilter(
        MissingThinkOpenTagFilter(),
        text_field=["reasoning", "output"],
        score_type=bool,
    ),
    ScoreFilter(
        NonEnglishFilter(args.tokenizer, args.lang_id_model_path),
        text_field=["system_prompt", "input", "output"],
        score_type=bool,
    ),
    ScoreFilter(
        TokenCountFilter(args.tokenizer, args.max_token_count),
        text_field=["system_prompt", "input", "output"],
        score_field="token_count",
        score_type=int,
    ),
    ScoreFilter(
        CompletionTokenCountFilter(args.tokenizer, args.max_completion_token_count),
        text_field=["output"],
        score_field="completion_token_count",
        score_type=int,
    ),
])

# Apply filters
dataset_df = filter_steps(dataset).df
print("Filtering data now...")
dataset_df = dataset_df.persist()

print(f"Filtered dataset has {dataset_df.npartitions} partitions")
print("Filtering complete!")


## Input/Output Reformatting

Reformat the input and output columns to use proper chat templates. This converts the structured conversation format into the tokenizer's expected format for training.


In [ ]:
print("Reformatting input and output columns...")

# Load tokenizer for chat template formatting
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer, use_fast=True)

# Update metadata for the reformed columns
meta = dataset_df._meta.copy()
meta["input"] = meta["input"].astype(str)

print(f"Reformatting input and output columns for {dataset_df.npartitions} partitions")
dataset_df = dataset_df.map_partitions(lambda df: format_partition(df, tokenizer), meta=meta)

dataset_df = dataset_df.persist()
print("Reformatting complete!")


## Post-Filtering Statistics

Calculate statistics after filtering to understand how many samples were removed during the filtering process.


In [ ]:
if args.generate_statistics:
    print("Generating post-filtering statistics...")
    current_count = dataset_df.shape[0].compute()
    if initial_count is not None:
        removed_count = initial_count - current_count
        print(f"Removed {removed_count:,} samples ({removed_count/initial_count*100:.1f}%)")
    print(f"Current number of samples: {current_count:,}")
else:
    print("Skipping post-filtering statistics generation for performance")


## Data Sorting

Sort the dataset by completion token count to enable curriculum learning. This arranges samples from shortest to longest completion length.


In [ ]:
print(f"Sorting dataset by completion_token_count...")
print(f"Dataset shape: {dataset_df.shape[0].compute():,} rows across {dataset_df.npartitions} partitions")

sorted_dataset_df = dataset_df.sort_values("completion_token_count")
sorted_dataset_df = sorted_dataset_df.persist()

print("Sorting complete!")


## Dataset Splitting

Split the dataset into reasoning ON and reasoning OFF subsets. This separation allows for balanced interleaving of the two types of reasoning samples.


In [ ]:
# Split into thinking ON and OFF
print("Splitting dataset into reasoning ON and OFF subsets...")
sorted_thinking_on = sorted_dataset_df.map_partitions(lambda df: df[df["reasoning"] == "on"])
sorted_thinking_off = sorted_dataset_df.map_partitions(lambda df: df[df["reasoning"] == "off"])

# Remove specified columns if requested
if args.remove_columns:
    print(f"Removing columns: {args.remove_columns}")
    sorted_thinking_on = sorted_thinking_on.drop(columns=args.remove_columns, axis=1)
    sorted_thinking_off = sorted_thinking_off.drop(columns=args.remove_columns, axis=1)

print("Dataset splitting complete!")


## Split Statistics

Calculate statistics for the reasoning ON and OFF subsets to understand the distribution of samples.


In [ ]:
if args.generate_statistics:
    print("Generating split statistics...")
    thinking_on_count = sorted_thinking_on.shape[0].compute()
    thinking_off_count = sorted_thinking_off.shape[0].compute()
    total_count = thinking_on_count + thinking_off_count
    
    print(f"Reasoning ON samples: {thinking_on_count:,} ({thinking_on_count/total_count*100:.1f}%)")
    print(f"Reasoning OFF samples: {thinking_off_count:,} ({thinking_off_count/total_count*100:.1f}%)")
    print(f"Total samples: {total_count:,}")
else:
    print("Skipping split statistics generation for performance")


## Dataset Interleaving

Interleave the reasoning ON and OFF datasets to create a balanced training set. Two interleaving modes are available:

- **Approximate interleaving** (default): Interleaves partitions for better performance
- **Global interleaving**: Interleaves individual rows for perfect balance (slower, CPU only)





In [ ]:
if not args.global_interleave:
    print("Performing approximate interleaving (partition-by-partition)...")
    # Interleave the sorted DataFrame partitions
    interleaved_df = interleave_partitions(sorted_thinking_on, sorted_thinking_off, gpu=args.device == "gpu")
else:
    if args.device == "gpu":
        raise RuntimeError("Global interleaving on GPU is not supported. Please use approximate interleaving or CPU.")
    
    print("Performing global interleaving (row-by-row)...")
    # Interleave the sorted DataFrame rows
    interleaved_df = interleave_rows(sorted_thinking_on, sorted_thinking_off, gpu=args.device == "gpu")

print("Dataset interleaving complete!")

## Final Statistics

Calculate final statistics for the processed dataset before saving.


In [ ]:
if args.generate_statistics:
    print("Generating final statistics...")
    final_count = interleaved_df.shape[0].compute()
    print(f"Final dataset size: {final_count:,}")
    
    if initial_count is not None:
        retention_rate = final_count / initial_count * 100
        print(f"Data retention rate: {retention_rate:.1f}%")
else:
    print("Skipping final statistics generation for performance")


## Save Dataset

Save the processed and interleaved dataset to the output directory as partitioned JSONL files.


In [ ]:
# Save dataset
print("Saving processed dataset...")
os.makedirs(args.output_dir, exist_ok=True)
output_path = os.path.join(args.output_dir, "part_*.jsonl")

interleaved_df.to_json(output_path, orient="records", lines=True)
print(f"Saved dataset to: {output_path}")

# Calculate and display total processing time
end_time = time.time()
total_time = end_time - start_time
print(f"Total processing time: {total_time:.2f} seconds ({total_time/60:.1f} minutes)")
print(f"Pipeline completed at {time.ctime()}")
!tree  /workspace/data/curated-data/

In [ ]:
## Stop Dask Cluster
client.cluster.close()
client.close()


## Summary

The data curation pipeline has completed successfully! Here's what was accomplished:

### Pipeline Steps:
1. **Data Loading**: Loaded JSONL files from input directory
2. **Filtering**: Applied 9 different filters to clean and validate data
3. **Reformatting**: Converted to proper chat template format
4. **Sorting**: Arranged by completion token count for curriculum learning
5. **Splitting**: Separated reasoning ON and OFF samples
6. **Interleaving**: Balanced the two reasoning types
7. **Saving**: Exported to partitioned JSONL files

### Usage Instructions:
1. **Configure**: Update the `Config` class with your paths and settings
2. **Run**: Execute cells sequentially from top to bottom
3. **Monitor**: Check statistics and progress messages
4. **Adjust**: Modify parameters as needed for your dataset

### Key Features:
- **Modular Design**: Each step can be run independently
- **Performance Optimized**: Uses Dask for distributed processing
- **Configurable**: Easy to adjust parameters and filters
- **Documented**: Clear markdown explanations for each step

The processed dataset is now ready for curriculum learning with balanced reasoning examples!
